In [4]:
# CELL 2: Visual Feature Extraction with CLIP
import os
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel

PROJECT_PATH = "Thesis_Data"

# 1. Load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP model on {device}...")
model_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def extract_clip_features(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 30
    
    frame_features = []
    count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Extract 1 frame per second
        if count % int(fps) == 0:
            # Convert BGR (OpenCV) to RGB (PIL)
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img)
            
            inputs = processor_clip(images=pil_img, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # --- THE BULLETPROOF FIX ---
                # 1. Run the vision part of the model explicitly (This outputs the "Object")
                vision_outputs = model_clip.vision_model(pixel_values=inputs['pixel_values'])
                
                # 2. Extract the raw tensor from inside that Object
                pooled_tensor = vision_outputs.pooler_output 
                
                # 3. Project it into the final 512-dimensional CLIP space
                features = model_clip.visual_projection(pooled_tensor)
                
            frame_features.append(features.cpu().numpy().flatten())
        count += 1
        
    cap.release()
    
    if not frame_features:
        return np.zeros(512) # CLIP patch32 outputs 512 dimensions
        
    # Average the features across the whole video
    return np.mean(frame_features, axis=0)

# 2. Process all videos
visual_features_dict_clip = {}
video_files = [f for f in os.listdir(f"{PROJECT_PATH}/videos") if f.endswith('.mp4')]

print("Extracting CLIP features (Visual)...")
for v_file in tqdm(video_files):
    v_id = v_file.replace(".mp4", "")
    v_path = f"{PROJECT_PATH}/videos/{v_file}"
    try:
        visual_features_dict_clip[v_id] = extract_clip_features(v_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/visual_features_clip.npy", visual_features_dict_clip)
print("CLIP Visual features saved successfully!")

Loading CLIP model on cpu...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting CLIP features (Visual)...


  0%|          | 0/486 [00:00<?, ?it/s]

CLIP Visual features saved successfully!


In [5]:
# CELL 3: Audio Feature Extraction with VGGishm

# 1. Load VGGish from PyTorch Hub
print("Loading VGGish model...")
# We use harritaylor's implementation which is the standard PyTorch port for VGGish
vggish = torch.hub.load('harritaylor/torchvggish', 'vggish')
vggish.eval()

def extract_vggish_features(audio_path):
    # The harritaylor VGGish port takes a wav file path directly,
    # automatically resamples it to 16kHz, computes the mel-spectrogram, 
    # and runs it through the network!
    with torch.no_grad():
        # Outputs shape: [number_of_seconds, 128]
        embeddings = vggish.forward(audio_path)
    
    # Average the embeddings over time to get one 128-D vector for the whole ad
    # Convert tensor to numpy
    features_np = embeddings.cpu().numpy()
    
    if len(features_np) == 0:
        return np.zeros(128)
        
    return np.mean(features_np, axis=0)

# 2. Process all audio files
audio_features_dict_vggish = {}
audio_files = [f for f in os.listdir(f"{PROJECT_PATH}/audio") if f.endswith('.wav')]

print("Extracting VGGish features (Audio)...")
for a_file in tqdm(audio_files):
    v_id = a_file.replace(".wav", "")
    a_path = f"{PROJECT_PATH}/audio/{a_file}"
    try:
        audio_features_dict_vggish[v_id] = extract_vggish_features(a_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/audio_features_vggish.npy", audio_features_dict_vggish)
print("VGGish Audio features saved successfully!")

Loading VGGish model...


c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\torch\hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/harritaylor/torchvggish/zipball/master" to C:\Users\yasam/.cache\torch\hub\master.zip
Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth" to C:\Users\yasam/.cache\torch\hub\checkpoints\vggish-10086976.pth


100%|██████████| 275M/275M [00:11<00:00, 25.4MB/s] 


Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish_pca_params-970ea276.pth" to C:\Users\yasam/.cache\torch\hub\checkpoints\vggish_pca_params-970ea276.pth


100%|██████████| 177k/177k [00:00<00:00, 3.55MB/s]

Extracting VGGish features (Audio)...



c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\torch\serialization.py:1832: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  result = unpickler.load()


  0%|          | 0/486 [00:00<?, ?it/s]

VGGish Audio features saved successfully!


In [9]:
# MASTER CELL: Data Loading, Deep Late Fusion, & Early Stopping
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. LOAD AND PREPARE DATA (Fixes the NameError)
# ==========================================
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)
visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()

X_visual, X_audio, y_labels = [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    if v_id in visual_dict and v_id in audio_dict:
        X_visual.append(visual_dict[v_id])
        X_audio.append(audio_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_visual)
X_audio = np.array(X_audio)
y_labels = np.array(y_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

# ==========================================
# 2. DEFINE THE DEEP LATE FUSION NETWORK
# ==========================================
class DeepLateFusionMLP(nn.Module):
    def __init__(self, visual_dim=512, audio_dim=128, num_classes=3):
        super(DeepLateFusionMLP, self).__init__()
        
        self.visual_net = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.BatchNorm1d(256), 
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        
        self.audio_net = nn.Sequential(
            nn.Linear(audio_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, visual_x, audio_x):
        v_feats = self.visual_net(visual_x)
        a_feats = self.audio_net(audio_x)
        combined = torch.cat((v_feats, a_feats), dim=1) 
        return self.classifier(combined)

# ==========================================
# 3. SETUP TRAINING & EARLY STOPPING
# ==========================================
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

all_true_labels = []
all_predictions = []
fold_accuracies = []

MAX_EPOCHS = 100
PATIENCE = 10  # Stop if validation loss doesn't improve for 10 epochs

print(f"\nStarting 5-Fold CV with Early Stopping (Patience: {PATIENCE})...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):
    print(f"\n--- FOLD {fold + 1}/{k_folds} ---")
    
    # Split and Scale Data
    X_v_train, X_v_val = X_visual[train_idx], X_visual[val_idx]
    X_a_train, X_a_val = X_audio[train_idx], X_audio[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    scaler_v = StandardScaler()
    X_v_train_scaled = scaler_v.fit_transform(X_v_train)
    X_v_val_scaled = scaler_v.transform(X_v_val)
    
    scaler_a = StandardScaler()
    X_a_train_scaled = scaler_a.fit_transform(X_a_train)
    X_a_val_scaled = scaler_a.transform(X_a_val)
    
    train_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_train_scaled, dtype=torch.float32), 
        torch.tensor(X_a_train_scaled, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    ), batch_size=32, shuffle=True)
    
    val_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_val_scaled, dtype=torch.float32), 
        torch.tensor(X_a_val_scaled, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    ), batch_size=32, shuffle=False)
    
    model = DeepLateFusionMLP(visual_dim=X_visual.shape[1], audio_dim=X_audio.shape[1], num_classes=len(le.classes_))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) 
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(MAX_EPOCHS):
        # -- TRAINING PHASE --
        model.train()
        for inputs_v, inputs_a, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs_v, inputs_a) 
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        # -- VALIDATION PHASE (For Early Stopping) --
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs_v, inputs_a, labels in val_loader:
                outputs = model(inputs_v, inputs_a)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
        val_loss /= len(val_loader)
        
        # -- EARLY STOPPING LOGIC --
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at Epoch {epoch + 1}! (Best Val Loss: {best_val_loss:.4f})")
            break
            
    # -- EVALUATE FOLD --
    model.eval()
    fold_preds, fold_labels = [], []
    with torch.no_grad():
        for inputs_v, inputs_a, labels in val_loader:
            outputs = model(inputs_v, inputs_a)
            _, preds = torch.max(outputs, 1)
            fold_preds.extend(preds.numpy())
            fold_labels.extend(labels.numpy())
            
    fold_acc = accuracy_score(fold_labels, fold_preds)
    fold_accuracies.append(fold_acc)
    print(f"Fold {fold + 1} Final Accuracy: {fold_acc:.4f}")
    
    all_true_labels.extend(fold_labels)
    all_predictions.extend(fold_preds)

print("\n" + "="*45)
print("=== FINAL DEEP LATE FUSION RESULTS ===")
print("="*45)
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})\n")
print("Detailed Classification Report:")
print(classification_report(all_true_labels, all_predictions, target_names=le.classes_))


Starting 5-Fold CV with Early Stopping (Patience: 10)...

--- FOLD 1/5 ---
Early stopping triggered at Epoch 14! (Best Val Loss: 1.0326)
Fold 1 Final Accuracy: 0.5444

--- FOLD 2/5 ---
Early stopping triggered at Epoch 15! (Best Val Loss: 0.9674)
Fold 2 Final Accuracy: 0.6404

--- FOLD 3/5 ---
Early stopping triggered at Epoch 14! (Best Val Loss: 1.0870)
Fold 3 Final Accuracy: 0.5955

--- FOLD 4/5 ---
Early stopping triggered at Epoch 13! (Best Val Loss: 1.0766)
Fold 4 Final Accuracy: 0.4719

--- FOLD 5/5 ---
Early stopping triggered at Epoch 14! (Best Val Loss: 1.0791)
Fold 5 Final Accuracy: 0.5056

=== FINAL DEEP LATE FUSION RESULTS ===
Average Accuracy: 0.5516 (+/- 0.0606)

Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.30      0.25      0.27        52
     Neutral       0.42      0.42      0.42       136
    Positive       0.66      0.68      0.67       258

    accuracy                           0.55       446
   macro 

In [12]:
# MASTER CELL: Binary Classification, Deep Late Fusion, & Early Stopping
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. LOAD AND PREPARE DATA (BINARY)
# ==========================================
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)

# --- THE BINARY FIX ---
# Combine Neutral and Negative into 'Non-Positive'
df['binary_sentiment'] = df['majority_sentiment'].apply(
    lambda x: 'Positive' if x == 'Positive' else 'Non-Positive'
)

visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()

X_visual, X_audio, y_labels = [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['binary_sentiment'] # Use the new binary label
    
    if v_id in visual_dict and v_id in audio_dict:
        X_visual.append(visual_dict[v_id])
        X_audio.append(audio_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_visual)
X_audio = np.array(X_audio)
y_labels = np.array(y_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

print(f"Data Loaded! Binary Distribution:")
unique, counts = np.unique(y_encoded, return_counts=True)
for i in range(len(unique)):
    print(f"{le.classes_[i]}: {counts[i]} videos")

# ==========================================
# 2. DEFINE THE DEEP LATE FUSION NETWORK
# ==========================================
class DeepLateFusionMLP(nn.Module):
    def __init__(self, visual_dim=512, audio_dim=128, num_classes=2): # Now defaults to 2 classes
        super(DeepLateFusionMLP, self).__init__()
        
        self.visual_net = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.BatchNorm1d(256), 
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        
        self.audio_net = nn.Sequential(
            nn.Linear(audio_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, visual_x, audio_x):
        v_feats = self.visual_net(visual_x)
        a_feats = self.audio_net(audio_x)
        combined = torch.cat((v_feats, a_feats), dim=1) 
        return self.classifier(combined)

# ==========================================
# 3. SETUP TRAINING & EARLY STOPPING
# ==========================================
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

all_true_labels = []
all_predictions = []
fold_accuracies = []

MAX_EPOCHS = 100
PATIENCE = 10  

print(f"\nStarting 5-Fold CV (Binary Classification)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):
    print(f"\n--- FOLD {fold + 1}/{k_folds} ---")
    
    X_v_train, X_v_val = X_visual[train_idx], X_visual[val_idx]
    X_a_train, X_a_val = X_audio[train_idx], X_audio[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    scaler_v = StandardScaler()
    X_v_train_scaled = scaler_v.fit_transform(X_v_train)
    X_v_val_scaled = scaler_v.transform(X_v_val)
    
    scaler_a = StandardScaler()
    X_a_train_scaled = scaler_a.fit_transform(X_a_train)
    X_a_val_scaled = scaler_a.transform(X_a_val)
    
    train_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_train_scaled, dtype=torch.float32), 
        torch.tensor(X_a_train_scaled, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    ), batch_size=32, shuffle=True)
    
    val_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_val_scaled, dtype=torch.float32), 
        torch.tensor(X_a_val_scaled, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    ), batch_size=32, shuffle=False)
    
    model = DeepLateFusionMLP(visual_dim=X_visual.shape[1], audio_dim=X_audio.shape[1], num_classes=len(le.classes_))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) 
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(MAX_EPOCHS):
        model.train()
        for inputs_v, inputs_a, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs_v, inputs_a) 
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs_v, inputs_a, labels in val_loader:
                outputs = model(inputs_v, inputs_a)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at Epoch {epoch + 1}! (Best Val Loss: {best_val_loss:.4f})")
            break
            
    model.eval()
    fold_preds, fold_labels = [], []
    with torch.no_grad():
        for inputs_v, inputs_a, labels in val_loader:
            outputs = model(inputs_v, inputs_a)
            _, preds = torch.max(outputs, 1)
            fold_preds.extend(preds.numpy())
            fold_labels.extend(labels.numpy())
            
    fold_acc = accuracy_score(fold_labels, fold_preds)
    fold_accuracies.append(fold_acc)
    print(f"Fold {fold + 1} Final Accuracy: {fold_acc:.4f}")
    
    all_true_labels.extend(fold_labels)
    all_predictions.extend(fold_preds)

print("\n" + "="*45)
print("=== FINAL BINARY LATE FUSION RESULTS ===")
print("="*45)
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})\n")
print("Detailed Classification Report:")
print(classification_report(all_true_labels, all_predictions, target_names=le.classes_))

Data Loaded! Binary Distribution:
Non-Positive: 188 videos
Positive: 258 videos

Starting 5-Fold CV (Binary Classification)...

--- FOLD 1/5 ---
Early stopping triggered at Epoch 12! (Best Val Loss: 0.6925)
Fold 1 Final Accuracy: 0.5556

--- FOLD 2/5 ---
Early stopping triggered at Epoch 14! (Best Val Loss: 0.6460)
Fold 2 Final Accuracy: 0.6742

--- FOLD 3/5 ---
Early stopping triggered at Epoch 13! (Best Val Loss: 0.6599)
Fold 3 Final Accuracy: 0.5955

--- FOLD 4/5 ---
Early stopping triggered at Epoch 12! (Best Val Loss: 0.6885)
Fold 4 Final Accuracy: 0.5281

--- FOLD 5/5 ---
Early stopping triggered at Epoch 13! (Best Val Loss: 0.6562)
Fold 5 Final Accuracy: 0.4831

=== FINAL BINARY LATE FUSION RESULTS ===
Average Accuracy: 0.5673 (+/- 0.0648)

Detailed Classification Report:
              precision    recall  f1-score   support

Non-Positive       0.49      0.47      0.48       188
    Positive       0.62      0.64      0.63       258

    accuracy                           0.57   